In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import seaborn as sns

**Introduction:**

During the sample period from October 13, 2021 – October 20, 2021, which included 50,061 visits to the homepage, just under 2% of users clicked “SHOP NOW”. This did attract about twice the number of users as the “iPhone” link in the left sidebar, but contrasts with the surrounding banner itself, which gathered a CTR of roughly 3.5%. We asked ourselves if it may be visual features of the “SHOP NOW” button or if the text may feel simply too immediate a commitment to buying for users.

**Primary objective: **

**Eniac intends to increase their sells of new iphone models. **

When visiting the website, an iphone picture occupies the centre of the website. Left to the picture of an iphone model, visitors can click the shop now button to choose an iphone.

Task:

Conduct an A/B test to gain knowledge about the performance of different "Shop Now" button designs. The following designs were tested:

1. White “SHOP NOW”

2. Red “SHOP NOW”

3. White “SEE DEALS”

4. Red “SEE DEALS”

Our relevant metrics are:

1. **Click-through-rate (CTR)** : The amount of clicks on the button divided by the total visits to the page. **The higher it is, the better.**

2. **Drop-off rate for the linked page** : Represents the percentage of visitors who initiate a conversion process such as a pruchase or sign up but do not complete it. It is an indicator to see how engaged visitors are at any point in the conversion process. **The lower it is, the better.**

3. **Homepage-return rate for the category pages:** Measures how often visitors return to the homepage after clicking the button. If visitors frequently return to the homepage, it suggests they might not be finding the desired information on the linked page.** The lower, the better.**


**Primary indicator for success: CTR**

**Method:**

**chi-square test**

Define the Null and alternative hypothesis:

**H0 = All versions have the same CTR**

**HA = There is a difference in the CTR for the different versions**

**Significance level:**

We are willing to accept a 5% chance of false positive (Type I Error)

In [38]:
alpha = 0.05

**Duration of the experiment:**

To determine the timeframe of the experiment, we used a power calculator (https://www.abtasty.com/sample-size-calculator/)

Using the standard **statistical power of 80**% along with a large **minimum detecteable effect of 20%.**

The test will have a 80% chance of detecting a difference in click through rates that is larger than 20%.

Consequence: As the MDE is this large, more subtle differences in the websites' performances will not reach a statistical significance. Advantage: We reduce the sample size required.

Current situation:

CTR at 2%, 7142 visits per day.

Based on the information given (CTR 2%, Minimum Detectable Effect : 20%, Statistical significance 95%, Average daily Visitors 7142 and our Number of Variations at 4, we need --> 19784 visitors per variation and a duration of 11 days.

The experiment ran between November 2, 2021 and November 16, 2021.

In [55]:
# Load experimental data
a = pd.read_csv("/content/eniac_a.csv")
b = pd.read_csv("/content/eniac_b.csv")
c = pd.read_csv("/content/eniac_c.csv")
d = pd.read_csv("/content/eniac_d.csv")

pd.set_option("display.max_colwidth", None)

a #25326 total visits
b # 24747
c #24876
d # 25233


,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,285,True,Homepage Version D - red SEE DEALS • https://eniac.com/index-d.php
1,25,div,mySidebar,305,True,"created 2021-10-27 • 14 days 0 hours 34 mins • 25233 visits, 22743 clicks"
2,4,a,Mac,274,True,NaN
3,69,a,iPhone,243,True,NaN
4,105,a,Accessories,1267,True,NaN
5,36,a,Chargers & Cables,1260,False,NaN
6,99,a,iPhone Accessories,1296,False,NaN
7,68,a,Watch Accessories,1252,False,NaN
8,13,a,Mac Accessories,1273,False,NaN
9,15,a,AirTag,201,False,NaN


In [56]:
# Getting useful datasets
a_1 = pd.DataFrame({
    "No Clicks": [a.loc[a["Name"] != "SHOP NOW", "No. clicks"].sum()],
    "Clicks": [a.loc[a["Name"] == "SHOP NOW", "No. clicks"].sum()],
    "Visits": [25326]})

b_1 = pd.DataFrame({
    "No Clicks": [b.loc[b["Name"] != "SHOP NOW", "No. clicks"].sum()],
    "Clicks": [b.loc[b["Name"] == "SHOP NOW", "No. clicks"].sum()],
    "Visits": [24747]})

c_1 = pd.DataFrame({
    "No Clicks": [c.loc[c["Name"] != "SEE DEALS", "No. clicks"].sum()],
    "Clicks": [c.loc[c["Name"] == "SEE DEALS", "No. clicks"].sum()],
    "Visits": [24876]})

d_1 = pd.DataFrame({
    "No Clicks": [d.loc[d["Name"] != "SEE DEALS", "No. clicks"].sum()],
    "Clicks": [d.loc[d["Name"] == "SEE DEALS", "No. clicks"].sum()],
    "Visits": [25233]
})

In [57]:
final_df = pd.concat([a_1, b_1, c_1, d_1], ignore_index=True)

final_df["Version"] = ["A", "B", "C", "D"]

final_df["CTR"] = final_df['Clicks'] / final_df['Visits']
final_df

,No Clicks,Clicks,Visits,Version,CTR
0,22662,512,25326,A,0.020216
1,22126,281,24747,B,0.011355
2,22504,527,24876,C,0.021185
3,22550,193,25233,D,0.007649


Conducting chi-squared test:

In [62]:
contingency_table = final_df[["Version", "No Clicks", "Clicks", "Visits"]]
contingency_table

,Version,No Clicks,Clicks,Visits
0,A,22662,512,25326
1,B,22126,281,24747
2,C,22504,527,24876
3,D,22550,193,25233


In [60]:
chi2, pvalue, dof, expected = chi2_contingency(contingency_table[["Clicks", "No Clicks"]])

print("Chi-squared:", chi2)
print("p-value:", pvalue)
print("Degrees of freedom:", dof)

Chi-squared: 213.3372405326989
p-value: 5.532250271124317e-46
Degrees of freedom: 3


In [39]:
if pvalue > alpha:
    print("The p-value is larger than alpha.")
else:
    print("The p-value is smaller than alpha.")

The p-value is smaller than alpha.


critical value = 7.815 (Chi)


Based on our Test, we can confidently reject the H0. There is a statistically relevant difference between the four versions' CTE.



**As we have more than 2 Variants, we have to perform post-hoc-test.**

Conducting more than one test increases the risk of type 1 error. In order to counter that, we will use the Bonferroni Adjustment.

We will conduct two additional tests:

1. We will test A and C to determine, if the CTR is statistically significant between them

2. We will compare A/C with B/D.

**Test 2: Comparing Version A and C**

H0 = There is no significant difference regarding the CTR.

HA = There is a significant difference regarding the CTR.

In [42]:
#Bonferroni-adjustment
alpha_2 = alpha/2

In [45]:
test2_df = final_df.loc[final_df["Version"].isin(["A", "C"]),
                        ["Clicks", "No Clicks"]]

chi2, pvalue, dof, expected = chi2_contingency(test2_df)
print("Chi-squared:", chi2)
print("p-value:", pvalue)
print("Degrees of freedom:", dof)

Chi-squared: 0.2918169433098661
p-value: 0.5890585361457614
Degrees of freedom: 1


In [46]:
if pvalue > alpha:
    print("The p-value is larger than alpha.")
else:
    print("The p-value is smaller than alpha.")

The p-value is larger than alpha.


We are unable to reject H0.

**Test 3: Comparing Group A/C with B/D**

In [52]:
ac = final_df.loc[final_df['Version'].isin(["A", "C"])]
bd = final_df.loc[final_df['Version'].isin(["B", "D"])]

Test_3 = [
    [ac["Clicks"].sum(), ac["No Clicks"].sum()],
    [bd["Clicks"].sum(), bd["No Clicks"].sum()]
]

chi2, pvalue, dof, expected = chi2_contingency(Test_3)
print("Chi-squared:", chi2)
print("p-value:", pvalue)
print("Degrees of freedom:", dof)

Chi-squared: 200.7685526586857
p-value: 1.4194433826124428e-45
Degrees of freedom: 1


In [53]:
if pvalue > alpha:
    print("The p-value is larger than alpha.")
else:
    print("The p-value is smaller than alpha.")

The p-value is smaller than alpha.


We can reject H0, there is a significant difference between Group A/C and B/D

Based on our tests, we know that either A or C is the best version.

We need to consider other indicators like the Drop-off-rate in order to identifiy the best version.